# L1 SCENIC regulon vs DEG hypergeometric enrichment
Preprocessing kept to the minimum:
1. parse `TargetGenes` from the L1 SCENIC ctx results (`/data1st1/junyi/output/sn0827L1/scenic_ctx/`)
   into a TF -> target-gene regulon table per region x L1 celltype (only TFs with any motif NES > 3);
2. load the L1 MAST DEG table and keep FDR < 0.05, split by Up/Down;
3. for every celltype x condition x TF, hypergeometric test of regulon targets against
   up/down DEGs (`hypergeom.sf(k-1, M, n, N)`), BH-corrected, saved to the L1 output folder.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import glob
import os
from ast import literal_eval

import numpy as np
import pandas as pd
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests

In [ ]:
# paths
ctx_dir = "/data1st1/junyi/output/sn0827L1/scenic_ctx/"
regulon_dir = "/data1st1/junyi/output/sn0827L1/tf_regulon/"
out_dir = "/data1st1/junyi/output/sn0827L1/tf_regulon_deg_hypergeom/"
deg_file = "/data2st2/junyi/output/hannan25_L1_mast_degs_fdr_merged_4conditions_7regions.csv"

NES_THRESHOLD = 3.0   # keep TargetGenes of motifs with NES > threshold
DEG_FDR = 0.05        # DEG significance cut-off
os.makedirs(regulon_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# preprocessing: ctx csv -> TF target-gene table per region x L1 celltype
# one TF can be represented by several motifs; targets of its significant motifs are unioned
def parse_enrichment_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return literal_eval(value)

regulon = {}  # fname -> {TF: set(target genes)}
for f in sorted(glob.glob(os.path.join(ctx_dir, "ctx_*.csv"))):
    fname = os.path.basename(f).replace("ctx_", "").replace(".csv", "")
    df = pd.read_csv(f)
    # ctx csv has a multi-row header: Enrichment.1 = NES, Enrichment.6 = TargetGenes
    df = df.rename(columns={"Unnamed: 0": "TF", "Unnamed: 1": "motif",
                            "Enrichment.1": "NES", "Enrichment.6": "TargetGenes"})
    df = df.dropna(subset=["TargetGenes"]).copy()
    df["NES"] = pd.to_numeric(df["NES"], errors="coerce")
    df = df[df["NES"] > NES_THRESHOLD]
    tf_targets = {}
    for tf, tg in zip(df["TF"], df["TargetGenes"]):
        genes = {g for g, _ in parse_enrichment_list(tg)}
        tf_targets.setdefault(tf, set()).update(genes)
    if not tf_targets:
        print(f"skip {fname}: no regulon with NES > {NES_THRESHOLD}")
        continue
    regulon[fname] = tf_targets
    pd.DataFrame({"TF": list(tf_targets),
                  "n_target": [len(v) for v in tf_targets.values()],
                  "targetgene": [",".join(sorted(v)) for v in tf_targets.values()]}
                 ).to_csv(os.path.join(regulon_dir, f"{fname}_tf_target.csv"), index=False)

print(f"{len(regulon)} cell types, e.g. {list(regulon)[:3]}")

In [ ]:
# load DEG table (L1 MAST, merged over 4 conditions x 7 regions), keep FDR < 0.05
df_deg = pd.read_csv(deg_file)
df_deg = df_deg[df_deg["FDR"] < DEG_FDR].copy()
df_deg["ctname"] = df_deg["Region"].astype(str) + "_" + df_deg["Class"].astype(str)
# compare gene symbols case-insensitively (ctx targets and DEG genes may differ in case)
df_deg["gene_up"] = df_deg["Gene"].astype(str).str.strip().str.upper()
df_deg = df_deg.drop_duplicates(["ctname", "Model", "gene_up"])
print("DEG rows:", len(df_deg), "| celltypes:", df_deg.ctname.nunique(), "| models:", sorted(df_deg.Model.unique()))

# DEG gene sets per celltype x Model x Direction
deg_sets = {
    (ct, model, d): set(g)
    for (ct, model, d), g in df_deg.groupby(["ctname", "Model", "Direction"])["gene_up"]
}

In [ ]:
# hypergeometric test: TF regulon targets vs up/down DEGs for each celltype x Model
#   M = background = union(regulon targets of this celltype, DEG genes of this celltype+Model)
#   n = regulon size, N = n up (or down) DEGs, k = overlap
rows = []
for fname, tf_targets in regulon.items():
    ct_deg = df_deg[df_deg.ctname == fname]
    if ct_deg.empty:
        print(f"skip {fname}: no DEGs")
        continue
    for (ct, model), g in ct_deg.groupby(["ctname", "Model"]):
        deg_up = deg_sets.get((ct, model, "Up"), set())
        deg_down = deg_sets.get((ct, model, "Down"), set())
        background = set().union(*tf_targets.values()) | deg_up | deg_down
        M = len(background)
        for tf, targets in tf_targets.items():
            tset = {gene.upper() for gene in targets}
            n = len(tset)
            k_up = len(tset & deg_up)
            k_down = len(tset & deg_down)
            rows.append({
                "ctname": ct, "Region": ct.split("_")[0], "Celltype": ct.split("_", 1)[1],
                "Model": model, "TF": tf, "n_target": n,
                "n_deg_up": len(deg_up), "n_deg_down": len(deg_down),
                "n_overlap_up": k_up, "n_overlap_down": k_down,
                "hypergeom_p_up": hypergeom.sf(k_up - 1, M, n, len(deg_up)) if k_up > 0 else 1.0,
                "hypergeom_p_down": hypergeom.sf(k_down - 1, M, n, len(deg_down)) if k_down > 0 else 1.0,
                "background_size": M,
                "overlap_up_genes": ",".join(sorted(tset & deg_up)),
                "overlap_down_genes": ",".join(sorted(tset & deg_down)),
            })

df_hyp = pd.DataFrame(rows)
df_hyp["hypergeom_FDR_up"] = multipletests(df_hyp["hypergeom_p_up"], method="fdr_bh")[1]
df_hyp["hypergeom_FDR_down"] = multipletests(df_hyp["hypergeom_p_down"], method="fdr_bh")[1]
df_hyp = df_hyp.sort_values(["ctname", "Model", "hypergeom_p_up", "hypergeom_p_down"]).reset_index(drop=True)
print(df_hyp.shape)
df_hyp.head()

In [ ]:
df_hyp.to_csv(os.path.join(out_dir, "L1_TF_regulon_deg_hypergeom.csv"), index=False)
sig = df_hyp[(df_hyp.hypergeom_FDR_up < 0.05) | (df_hyp.hypergeom_FDR_down < 0.05)]
sig.to_csv(os.path.join(out_dir, "L1_TF_regulon_deg_hypergeom_FDR05.csv"), index=False)
print("saved:", out_dir)
print(f"all tests: {len(df_hyp)} | significant (FDR<0.05): {len(sig)}")